# ❄️ Day 18: Q-Learning from Scratch on FrozenLake-v1
### 100 Days of Machine Learning | Syed Muhammad Qamran
**Topic:** Reinforcement Learning (Model-Free, Off-Policy TD Control)  
**Environment:** Gymnasium `FrozenLake-v1` (4x4 Grid)

---

### Key Theory:
The Bellman Optimality Equation for Q-Learning:
$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha \left[ R_{t+1} + \gamma \max_{a'} Q(S_{t+1}, a') - Q(S_t, A_t) \right]$$
- **Off-Policy:** Learns the optimal greedy policy regardless of which exploratory action the agent actually took.
- **Epsilon-Greedy Exploration:** Balances exploring unvisited grid holes vs. exploiting high-reward paths to the Frisbee goal.


In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
%matplotlib inline

print("Gymnasium and NumPy ready!")


In [ ]:
class QLearningAgent:
    """Model-free Q-Learning agent with epsilon decay and random tie-breaking."""
    def __init__(self, n_states=16, n_actions=4, alpha=0.8, gamma=0.95, epsilon=1.0, epsilon_min=0.01, epsilon_decay=0.999):
        self.n_states = n_states
        self.n_actions = n_actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.Q = np.zeros((n_states, n_actions), dtype=np.float64)

    def get_action(self, state, greedy=False):
        if not greedy and np.random.uniform(0, 1) < self.epsilon:
            return np.random.randint(self.n_actions)
        max_q = np.max(self.Q[state, :])
        best = np.where(self.Q[state, :] == max_q)[0]
        return int(np.random.choice(best))

    def update(self, state, action, reward, next_state):
        td_target = reward + self.gamma * np.max(self.Q[next_state, :])
        td_error = td_target - self.Q[state, action]
        self.Q[state, action] += self.alpha * td_error
        return td_error

    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
        return self.epsilon


In [ ]:
# Training Q-Learning on FrozenLake-v1 (Slippery = False for deterministic path demonstration)
env = gym.make("FrozenLake-v1", is_slippery=False)
agent = QLearningAgent(alpha=0.8, gamma=0.95, epsilon=1.0, epsilon_decay=0.998)

num_episodes = 1000
rewards = []
success_rates = []

for ep in range(num_episodes):
    state, _ = env.reset()
    total_r = 0
    for _ in range(100):
        action = agent.get_action(state, greedy=False)
        next_state, reward, terminated, truncated, _ = env.step(action)
        agent.update(state, action, reward, next_state)
        state = next_state
        total_r += reward
        if terminated or truncated:
            break
    agent.decay_epsilon()
    rewards.append(total_r)
    success_rates.append(np.mean(rewards[-min(100, len(rewards)):]) * 100)

print(f"Training complete! Final 100-episode success rate: {success_rates[-1]:.1f}%")

# Plot Learning Curve
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(success_rates, color="#10B981", lw=2)
ax.set_title("FrozenLake Q-Learning Success Rate (100-Episode Rolling Avg)", fontsize=13, fontweight='bold')
ax.set_xlabel("Episode")
ax.set_ylabel("Success Rate (%)")
plt.show()


In [ ]:
# Visualize Learned Value Function & Optimal Actions (Arrows)
action_arrows = {0: "←", 1: "↓", 2: "→", 3: "↑"}
state_values = np.max(agent.Q, axis=1).reshape((4, 4))
best_actions = np.argmax(agent.Q, axis=1).reshape((4, 4))

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(state_values, annot=True, fmt=".2f", cmap="Blues", cbar=True, ax=ax)
for r in range(4):
    for c in range(4):
        arrow = action_arrows[best_actions[r, c]]
        ax.text(c + 0.5, r + 0.8, arrow, ha='center', va='center', color='red', fontsize=18, fontweight='bold')
ax.set_title("Optimal State Values & Policy Directions", fontsize=12, fontweight='bold')
plt.show()
